# ShieldWise Evaluation Workflow

This notebook brings together the evaluation evidence for my `ShieldWise` insurance claim fraud detection project.

I have kept this notebook aligned with the version of the project that is actually implemented in the repository. The main parts of the submitted system are:

- a FastAPI backend for claim intake, scoring, dashboards, and alerts
- a React frontend for the public page, policyholder dashboard, and investigator dashboard
- an NLP component for claim-language risk scoring
- supporting receipt-evidence model artefacts
- regression tests for the insurance workflow in `tests/test_api_insurance.py`

The aim of this notebook is to show the evidence that already exists in the project. I have not added extra claims about models that are not actually used or stored in the repository.

## Scope of the Submission

For this submission, the main working product is the insurance workflow. The most important folders and files are:

- `src/api/`
- `src/frontend/`
- `tests/test_api_insurance.py`

The evaluation evidence I use in this notebook comes from:

- `backend/saved_models/nlp_metrics.json`
- `backend/receipts_models/cv_metrics.json`
- the retained notebooks and model artefacts in `backend/`

One important point is that the live API uses a lighter document-checking process at runtime, while the repository also keeps the deeper CV model work as supporting research evidence. I have made that distinction clear so the evaluation does not overstate what is deployed in the running application.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "backend":
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_DIR = PROJECT_ROOT / "backend"
NLP_METRICS_PATH = BACKEND_DIR / "saved_models" / "nlp_metrics.json"
CV_METRICS_PATH = BACKEND_DIR / "receipts_models" / "cv_metrics.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"NLP metrics path: {NLP_METRICS_PATH}")
print(f"CV metrics path: {CV_METRICS_PATH}")

In [ ]:
with open(NLP_METRICS_PATH, "r", encoding="utf-8") as file:
    nlp_metrics = json.load(file)

with open(CV_METRICS_PATH, "r", encoding="utf-8") as file:
    cv_metrics = json.load(file)

nlp_metrics, cv_metrics

## NLP Evaluation Summary

The NLP part of the project supports the claim-language risk score. In simple terms, it checks whether the wording of a claim message looks more like a normal claim or a suspicious message.

The saved metrics are useful evidence for the report because they show that the claim-email model pipeline was trained and evaluated. However, I would still treat these results carefully. The dataset is small, so I describe this as a prototype evaluation rather than proof that the model is ready for real insurance deployment.

In [ ]:
nlp_rows = []
for model_name, model_metrics in nlp_metrics["models"].items():
    nlp_rows.append({"model": model_name, **model_metrics})

nlp_df = pd.DataFrame(nlp_rows).set_index("model")
nlp_df

In [ ]:
summary_df = pd.DataFrame([
    {
        "dataset_path": nlp_metrics["dataset_path"],
        "rows": nlp_metrics["rows"],
        "train_rows": nlp_metrics["train_rows"],
        "test_rows": nlp_metrics["test_rows"],
        "selected_features": nlp_metrics["selected_features"],
    }
])
summary_df

In [ ]:
ax = nlp_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot(
    kind="bar",
    figsize=(10, 5),
    ylim=(0, 1.05),
    title="ShieldWise NLP Model Metrics"
)
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Receipt Evidence Model Evaluation Summary

The repository also includes supporting metrics for the receipt-evidence models. I use these results as background evidence that computer vision work was completed for the project.

At the same time, the current live insurance workflow uses the lighter checks in `src/api/services/document_risk.py`. Because of that, I present the receipt-model metrics as supporting model evidence, not as a claim that those deep models are fully served in the live API.

In [ ]:
cv_rows = []
for model_name, model_metrics in cv_metrics.items():
    cv_rows.append({"model": model_name, **model_metrics})

cv_df = pd.DataFrame(cv_rows).set_index("model")
cv_df

In [ ]:
plot_columns = [column for column in ["accuracy", "precision", "recall", "f1_score", "roc_auc"] if column in cv_df.columns]
ax = cv_df[plot_columns].fillna(0).plot(
    kind="bar",
    figsize=(10, 5),
    ylim=(0, 1.05),
    title="ShieldWise Supporting Receipt Model Metrics"
)
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## How These Results Fit My Final Submission

For my final write-up, I would explain the evaluation in three layers:

1. **Active application workflow**
   The working system I am submitting is the insurance platform implemented in `src/api/`, `src/frontend/`, and `tests/test_api_insurance.py`.

2. **Retained model evidence**
   The NLP and receipt-model metrics show that the repository includes modelling work connected to claim-language scoring and receipt evidence analysis.

3. **Runtime distinction**
   The current API document-risk flow is lightweight at runtime. Therefore, the report should not say that the deep receipt models are directly deployed in the live application unless that is made true in the implementation.

## Wording I Can Use In The Report

A clear way to explain this in the report would be:

> In this project, I developed ShieldWise as an insurance claim fraud detection workflow. The submitted system includes a FastAPI backend, a React frontend, claim submission, dashboard views, evidence upload, and regression tests. The repository also keeps trained NLP and receipt-analysis artefacts as supporting model evidence. I therefore use the model metrics to support the evaluation, while making it clear that the current runtime document-risk checks are lighter than the retained CV research models.

This wording is honest about what the project does, while still showing the modelling and engineering work that supports the final system.

## Limitations I Should State Clearly

- The retained metrics are useful supporting evidence, but they are not a full production evaluation package.
- The repository includes supporting CV model outputs, but the active runtime evidence scoring path is lighter and rule-based.
- The NLP metrics are very strong on the retained dataset, so I should avoid claiming that the model will generalise perfectly to real insurance data without more testing.
- The final evaluation section should mention both the automated workflow tests and the retained model metrics, because the project combines software engineering with supporting AI model work.

In [ ]:
evaluation_inventory = pd.DataFrame([
    {
        "artifact": "NLP metrics",
        "path": str(NLP_METRICS_PATH.relative_to(PROJECT_ROOT)),
        "role_in_submission": "Supporting evidence for claim-language scoring"
    },
    {
        "artifact": "Receipt CV metrics",
        "path": str(CV_METRICS_PATH.relative_to(PROJECT_ROOT)),
        "role_in_submission": "Supporting evidence for retained receipt-model research assets"
    },
    {
        "artifact": "Insurance API regression tests",
        "path": "tests/test_api_insurance.py",
        "role_in_submission": "Primary verification of the active end-to-end workflow"
    }
])
evaluation_inventory